## Импорт и построение графа

In [1]:
import sys
sys.path.append(".")  # чтобы импортировать из src/

from src.graph_builder import build_graph
import networkx as nx
import matplotlib.pyplot as plt

G = build_graph()
print(f"Узлов: {G.number_of_nodes()}")
print(f"Рёбер: {G.number_of_edges()}")
print(f"Плотность: {nx.density(G):.4f}")

ImportError: cannot import name 'build_graph' from 'src.graph_builder' (c:\Users\Mihail\Desktop\ДЗ\5сем\Курсач\src\graph_builder.py)

### Визуализация графа (упрощённая)

In [ ]:
# Берём подграф для наглядности (иначе 277 узлов — каша)
# Показываем только узлы с высокой степенью связности
degrees = dict(G.degree())
top_nodes = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:30]
top_node_ids = [n[0] for n in top_nodes]

H = G.subgraph(top_node_ids)

fig, ax = plt.subplots(figsize=(14, 10))
pos = {node: (G.nodes[node]['lng'], G.nodes[node]['lat']) for node in H.nodes()}
nx.draw(H, pos, ax=ax, with_labels=True, 
        node_size=300, node_color='lightblue',
        font_size=7, arrows=True, arrowsize=10)
ax.set_title("Топ-30 остановок по связности")
plt.tight_layout()
plt.show()

### Визуализация на карте (folium): 

In [ ]:
import folium

# Центрируем карту на Чебоксарах
m = folium.Map(location=[56.1322, 47.2519], zoom_start=12)

# Добавляем все остановки
for node_id, data in G.nodes(data=True):
    folium.CircleMarker(
        location=[data['lat'], data['lng']],
        radius=3,
        popup=data['name'],
        color='blue',
        fill=True,
        fill_opacity=0.6
    ).add_to(m)

# Добавляем рёбра (линии маршрутов)
for u, v, data in G.edges(data=True):
    folium.PolyLine(
        locations=[[G.nodes[u]['lat'], G.nodes[u]['lng']],
                   [G.nodes[v]['lat'], G.nodes[v]['lng']]],
        color='gray',
        weight=1,
        opacity=0.4
    ).add_to(m)

m.save("graph_map.html")
print("Карта сохранена в graph_map.html")
m

#### Тест Дейкстры (поиск кратчайшего маршрута):

In [ ]:
# Находим кратчайший путь между двумя остановками
source = 235  # "Перова"
target = 415  # "Альгешево"

try:
    path = nx.dijkstra_path(G, source, target, weight='weight')
    path_length = nx.dijkstra_path_length(G, source, target, weight='weight')
    
    print(f"Кратчайший путь: {path_length:.0f} м")
    print(f"Остановки маршрута:")
    for node in path:
        print(f"  - {G.nodes[node]['name']}")
except nx.NetworkXNoPath:
    print("Путь не найден!")